# 📰 Phase 2: Article Content Scraper — LPDP

---

## 🎯 Tujuan Notebook
Mengambil **isi lengkap artikel** dari 1.370 artikel LPDP yang sudah divalidasi dan dilabeli secara manual.

### Alur Proses:
| Langkah | Deskripsi |
|---------|----------|
| 1️⃣ Load Dataset | Muat `dataset_lpdp_manual.csv` (1.937 total, 1.370 valid) |
| 2️⃣ Decode URL | Konversi Google News URL → URL artikel asli via `googlenewsdecoder` |
| 3️⃣ Scrape Content | Ekstrak isi teks artikel via `newspaper3k` secara paralel |
| 4️⃣ Export | Simpan ke `dataset_lpdp_konten_raw.csv` |

> 💾 **Resume-Friendly**: Notebook ini akan **melanjutkan** proses jika sebagian artikel sudah di-decode/scrape sebelumnya.
> Cukup jalankan ulang — artikel yang sudah ada contentnya tidak akan diproses ulang.

---

**Kelompok 5:** Amel, Celine, Iqbal, Nida, Salwa 

**PIC Phase 2:** Amel  

**Input:** `dataset_lpdp_manual.csv` — 1.937 total, 1.370 valid, berlabel (Positive: 462 / Neutral: 506 / Negative: 402)  

**Output:** `dataset_lpdp_konten_raw.csv` — artikel dengan konten lengkap hasil scraping

## 📦 Import Libraries

Library yang digunakan:
- **pandas**: Manipulasi DataFrame
- **newspaper3k**: Ekstrak teks artikel dari URL
- **googlenewsdecoder**: Decode Google News redirect URL → URL asli
- **concurrent.futures**: ThreadPoolExecutor untuk scraping paralel
- **tqdm**: Progress bar responsif di Jupyter
- **time**: Delay antar batch (menghindari rate-limit)

In [1]:
import pandas as pd
import time
import os
import sys
from newspaper import Article
from concurrent.futures import ThreadPoolExecutor, as_completed
from googlenewsdecoder import new_decoderv1
from tqdm.auto import tqdm

print("✅ Semua library berhasil diimport!")
print(f"   pandas       : {pd.__version__}")
print(f"   tqdm         : ok")
print(f"   newspaper3k  : ok")
print(f"   googlenewsdecoder : ok")

✅ Semua library berhasil diimport!
   pandas       : 3.0.2
   tqdm         : ok
   newspaper3k  : ok
   googlenewsdecoder : ok


c:\Users\mikba\Downloads\Documents\PBA\PBA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 📂 Load Dataset

Dataset diambil dari `dataset_lpdp_manual.csv` (1.370 artikel valid dari 1.937 total).

> 💾 **Resume**: Jika `dataset_lpdp_konten_raw.csv` sudah ada dari run sebelumnya, progress decode & scrape akan dilanjutkan otomatis — artikel yang sudah selesai tidak akan diproses ulang.

In [2]:
SOURCE_FILE = "dataset_lpdp_manual.csv"
OUTPUT_FILE = "dataset_lpdp_konten_raw.csv"

# === Selalu mulai dari source lengkap (1.370 artikel valid) ===
df_all   = pd.read_csv(SOURCE_FILE)
df_valid = df_all[df_all["Valid?"] == True].copy().reset_index(drop=True)
print(f"📂 Source: {SOURCE_FILE}")
print(f"   Total artikel  : {len(df_all)}")
print(f"   Artikel valid  : {len(df_valid)}")

# === Pastikan kolom wajib ada ===
if "Actual_URL" not in df_valid.columns:
    df_valid["Actual_URL"] = None
if "Content" not in df_valid.columns:
    df_valid["Content"] = None

# === Resume dari progress sebelumnya jika ada ===
if os.path.exists(OUTPUT_FILE):
    df_resume = pd.read_csv(OUTPUT_FILE)
    cols_to_resume = [c for c in ["Actual_URL", "Content"] if c in df_resume.columns]
    if cols_to_resume:
        df_resume_sub = df_resume[["URL"] + cols_to_resume].dropna(subset=["URL"])
        df_merged = df_valid.merge(df_resume_sub, on="URL", how="left", suffixes=("", "_r"))
        n_dec_resumed, n_cnt_resumed = 0, 0
        for col in cols_to_resume:
            rcol = col + "_r"
            if rcol in df_merged.columns:
                mask = df_merged[col].isna() & df_merged[rcol].notna()
                df_merged.loc[mask, col] = df_merged.loc[mask, rcol]
                df_merged.drop(columns=[rcol], inplace=True)
                if col == "Actual_URL": n_dec_resumed = mask.sum()
                if col == "Content":    n_cnt_resumed = mask.sum()
        df_valid = df_merged
        print(f"\n♻️  Resume dari: {OUTPUT_FILE}")
        print(f"   URL decode di-carry over : {n_dec_resumed}")
        print(f"   Content di-carry over    : {n_cnt_resumed}")

# === Summary status saat ini ===
n_total       = len(df_valid)
n_decoded     = df_valid["Actual_URL"].notna().sum()
n_scraped     = df_valid["Content"].notna().sum()
n_pending_dec = n_total - n_decoded
n_pending_scr = df_valid[(df_valid["Content"].isna()) & (df_valid["Actual_URL"].notna())].shape[0]

print(f"\n{'='*55}")
print("📊 STATUS DATASET SAAT INI")
print(f"{'='*55}")
print(f"  Total artikel valid    : {n_total}")
print(f"  URL sudah di-decode    : {n_decoded} ({n_decoded/n_total*100:.1f}%)")
print(f"  Content sudah di-scrape: {n_scraped} ({n_scraped/n_total*100:.1f}%)")
print(f"  URL masih perlu decode : {n_pending_dec}")
print(f"  Artikel masih perlu scrape: {n_pending_scr}")
print(f"{'='*55}")

📂 Source: dataset_lpdp_manual.csv
   Total artikel  : 1937
   Artikel valid  : 1370

📊 STATUS DATASET SAAT INI
  Total artikel valid    : 1370
  URL sudah di-decode    : 0 (0.0%)
  Content sudah di-scrape: 0 (0.0%)
  URL masih perlu decode : 1370
  Artikel masih perlu scrape: 0


## 🔓 Phase 1 — Decode Google News URLs

Google News menggunakan **redirect URL** (`news.google.com/rss/articles/...`) yang harus di-decode dulu
sebelum bisa di-scrape.

### Konfigurasi:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `DECODE_WORKERS` | 4 | Thread parallel (lebih tinggi → rate-limit lebih cepat) |
| `DECODE_BATCH` | 50 | Batch per iterasi |
| Delay antar batch | 2 detik | Menghindari rate-limit Google News |
| Max retry | 2x | Retry sekali jika gagal |

> ⚠️ **Penting**: Google News memiliki rate-limit. Jika terlalu banyak request dalam waktu singkat,
> decode akan gagal. Delay 2 detik antar batch adalah langkah pencegahan.

In [3]:
def safe_decode_url(args):
    """Decode satu Google News URL → URL asli. Retry 1x jika gagal."""
    idx, url = args
    for attempt in range(2):
        try:
            result = new_decoderv1(url)
            if result.get("status"):
                return idx, result["decoded_url"]
        except Exception:
            if attempt == 0:
                time.sleep(0.5)
    return idx, None


# Kumpulkan URL yang belum di-decode
need_decode = [
    (i, df_valid.loc[i, "URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Actual_URL"])
]

DECODE_WORKERS = 4
DECODE_BATCH   = 50

print(f"🔓 URL yang perlu di-decode: {len(need_decode)} dari {len(df_valid)} total\n")

if len(need_decode) == 0:
    print("✅ Semua URL sudah di-decode sebelumnya. Skip fase ini.")
else:
    decode_success = 0
    decode_failed  = 0
    total_batches  = (len(need_decode) + DECODE_BATCH - 1) // DECODE_BATCH

    # Progress bar keseluruhan
    pbar_total = tqdm(
        total=len(need_decode),
        desc="🔓 Decode URL",
        unit="url",
        colour="blue"
    )

    for batch_num, batch_start in enumerate(range(0, len(need_decode), DECODE_BATCH), 1):
        batch = need_decode[batch_start:batch_start + DECODE_BATCH]

        with ThreadPoolExecutor(max_workers=DECODE_WORKERS) as executor:
            futures = {executor.submit(safe_decode_url, item): item for item in batch}
            for future in as_completed(futures):
                idx, decoded = future.result()
                if decoded:
                    df_valid.loc[idx, "Actual_URL"] = decoded
                    decode_success += 1
                else:
                    decode_failed += 1
                pbar_total.set_postfix({
                    "✅ OK": decode_success,
                    "❌ Gagal": decode_failed
                })
                pbar_total.update(1)

        # Simpan progress setelah setiap batch
        df_valid.to_csv(OUTPUT_FILE, index=False)

        # Delay antar batch (kecuali batch terakhir)
        if batch_start + DECODE_BATCH < len(need_decode):
            time.sleep(2)

    pbar_total.close()

    print(f"\n{'='*55}")
    print("📊 HASIL DECODE URL")
    print(f"{'='*55}")
    print(f"  Diproses   : {len(need_decode)}")
    print(f"  ✅ Berhasil : {decode_success} ({decode_success/len(need_decode)*100:.1f}%)")
    print(f"  ❌ Gagal    : {decode_failed} ({decode_failed/len(need_decode)*100:.1f}%)")
    total_decoded = df_valid["Actual_URL"].notna().sum()
    print(f"  Total decoded (kumulatif): {total_decoded}/{len(df_valid)}")
    print(f"{'='*55}")
    print(f"\n💾 Progress disimpan ke: {OUTPUT_FILE}")

🔓 URL yang perlu di-decode: 1370 dari 1370 total



🔓 Decode URL: 100%|██████████| 1370/1370 [42:04<00:00,  1.84s/url, ✅ OK=1305, ❌ Gagal=65] 


📊 HASIL DECODE URL
  Diproses   : 1370
  ✅ Berhasil : 1305 (95.3%)
  ❌ Gagal    : 65 (4.7%)
  Total decoded (kumulatif): 1305/1370

💾 Progress disimpan ke: dataset_lpdp_konten_raw.csv


## 📰 Phase 2 — Scrape Isi Artikel

Setelah mendapatkan URL asli, kini kita ekstrak **isi lengkap teks** dari setiap artikel menggunakan `newspaper3k`.

### Konfigurasi:
| Parameter | Nilai | Keterangan |
|-----------|-------|------------|
| `SCRAPE_WORKERS` | 20 | Thread paralel (I/O-bound, aman untuk tinggi) |
| `SCRAPE_BATCH` | 200 | Batch per iterasi |
| Timeout per artikel | 8 detik | Artikel lambat akan di-skip |
| Min text length | 50 karakter | Filter noise & halaman error |

> 💡 **Mengapa 20 workers?** Scraping artikel adalah *I/O-bound task* (menunggu response server).
> Menggunakan banyak thread sangat efektif karena thread lain bisa berjalan saat satu thread menunggu.
>
> 💾 **Auto-save**: Dataset disimpan otomatis setelah setiap batch selesai.

In [4]:
def scrape_content(args):
    """Scrape isi teks artikel dari URL. Kembalikan None jika gagal atau terlalu pendek."""
    idx, url = args
    if url is None or (isinstance(url, float) and pd.isna(url)):
        return idx, None
    try:
        article = Article(url, language="id", request_timeout=8)
        article.download()
        article.parse()
        text = article.text.strip()
        return idx, text if len(text) > 50 else None
    except Exception:
        return idx, None


# Kumpulkan artikel yang belum di-scrape (sudah punya Actual_URL tapi belum ada Content)
need_scrape = [
    (i, df_valid.loc[i, "Actual_URL"])
    for i in range(len(df_valid))
    if pd.isna(df_valid.loc[i, "Content"]) and pd.notna(df_valid.loc[i, "Actual_URL"])
]

SCRAPE_WORKERS = 20
SCRAPE_BATCH   = 200

print(f"📰 Artikel yang perlu di-scrape: {len(need_scrape)} dari {len(df_valid)} total\n")

if len(need_scrape) == 0:
    print("✅ Semua artikel sudah di-scrape sebelumnya. Skip fase ini.")
else:
    scrape_success = 0
    scrape_failed  = 0

    # Progress bar keseluruhan
    pbar_scrape = tqdm(
        total=len(need_scrape),
        desc="📰 Scrape Content",
        unit="artikel",
        colour="green"
    )

    for batch_start in range(0, len(need_scrape), SCRAPE_BATCH):
        batch = need_scrape[batch_start:batch_start + SCRAPE_BATCH]

        with ThreadPoolExecutor(max_workers=SCRAPE_WORKERS) as executor:
            futures = {executor.submit(scrape_content, item): item for item in batch}
            for future in as_completed(futures):
                idx, content = future.result()
                if content:
                    df_valid.loc[idx, "Content"] = content
                    scrape_success += 1
                else:
                    scrape_failed += 1
                pbar_scrape.set_postfix({
                    "✅ Content": scrape_success,
                    "❌ Kosong": scrape_failed
                })
                pbar_scrape.update(1)

        # Auto-save setelah setiap batch
        df_valid.to_csv(OUTPUT_FILE, index=False)

    pbar_scrape.close()

    print(f"\n{'='*55}")
    print("📊 HASIL SCRAPING CONTENT")
    print(f"{'='*55}")
    print(f"  Diproses    : {len(need_scrape)}")
    print(f"  ✅ Berhasil  : {scrape_success} ({scrape_success/len(need_scrape)*100:.1f}%)")
    print(f"  ❌ Kosong/Gagal: {scrape_failed} ({scrape_failed/len(need_scrape)*100:.1f}%)")
    total_content = df_valid["Content"].notna().sum()
    print(f"  Total content (kumulatif): {total_content}/{len(df_valid)}")
    print(f"{'='*55}")
    print(f"\n💾 Progress disimpan ke: {OUTPUT_FILE}")

📰 Artikel yang perlu di-scrape: 1305 dari 1370 total



📰 Scrape Content: 100%|██████████| 1305/1305 [07:59<00:00,  2.72artikel/s, ✅ Content=1038, ❌ Kosong=267]


📊 HASIL SCRAPING CONTENT
  Diproses    : 1305
  ✅ Berhasil  : 1038 (79.5%)
  ❌ Kosong/Gagal: 267 (20.5%)
  Total content (kumulatif): 1038/1370

💾 Progress disimpan ke: dataset_lpdp_konten_raw.csv


## 🔍 Quality Check & Preview Data

Validasi kualitas hasil scraping:
- Coverage content (persentase artikel berhasil di-scrape)
- Distribusi panjang konten
- Sample artikel berhasil & gagal
- Distribusi per label sentimen

In [5]:
print("="*60)
print("📊 QUALITY CHECK — HASIL SCRAPING KONTEN")
print("="*60)

# === Coverage ===
n_total   = len(df_valid)
n_content = df_valid["Content"].notna().sum()
n_decoded = df_valid["Actual_URL"].notna().sum()
coverage  = n_content / n_total * 100

print(f"\n🔢 Coverage:")
print(f"   Total artikel valid   : {n_total}")
print(f"   URL berhasil di-decode: {n_decoded} ({n_decoded/n_total*100:.1f}%)")
print(f"   Artikel punya content : {n_content} ({coverage:.1f}%)")

# === Distribusi panjang content ===
df_valid["content_len"] = df_valid["Content"].str.len()
print(f"\n📏 Distribusi Panjang Content (karakter):")
print(df_valid["content_len"].describe().round(0).to_string())

# === Distribusi per sentimen ===
print(f"\n🏷️  Coverage per Label Sentimen:")
print(f"{'Label':<12} {'Total':>7} {'Ada Content':>12} {'Coverage':>10}")
print("-"*45)
for label in ["Positive", "Neutral", "Negative"]:
    df_label = df_valid[df_valid["Sentiment"] == label]
    n_lbl   = len(df_label)
    n_lbl_c = df_label["Content"].notna().sum()
    if n_lbl > 0:
        print(f"{label:<12} {n_lbl:>7} {n_lbl_c:>12} {n_lbl_c/n_lbl*100:>9.1f}%")
print("-"*45)

# === Sample artikel berhasil ===
print(f"\n👁️  Sample Artikel Berhasil Scrape (3 Pertama):")
sample_ok = df_valid[df_valid["Content"].notna()][["Title", "Publisher", "Sentiment", "content_len"]].head(3)
for _, row in sample_ok.iterrows():
    print(f"   [{row['Sentiment']}] {row['Title'][:60]}... ({int(row['content_len'])} chars) — {row['Publisher']}")

# === Sample artikel gagal ===
n_failed = df_valid["Content"].isna().sum()
if n_failed > 0:
    print(f"\n❌ Sample Artikel GAGAL Scrape ({min(n_failed, 3)} dari {n_failed}):")
    sample_fail = df_valid[df_valid["Content"].isna()][["Title", "Publisher", "Sentiment"]].head(3)
    for _, row in sample_fail.iterrows():
        print(f"   [{row['Sentiment']}] {row['Title'][:60]}... — {row['Publisher']}")

print("\n" + "="*60)

📊 QUALITY CHECK — HASIL SCRAPING KONTEN

🔢 Coverage:
   Total artikel valid   : 1370
   URL berhasil di-decode: 1305 (95.3%)
   Artikel punya content : 1038 (75.8%)

📏 Distribusi Panjang Content (karakter):
count     1038.0
mean      3275.0
std       2062.0
min        143.0
25%       2044.0
50%       2700.0
75%       3828.0
max      17306.0

🏷️  Coverage per Label Sentimen:
Label          Total  Ada Content   Coverage
---------------------------------------------
Positive         462          385      83.3%
Neutral          506          342      67.6%
Negative         402          311      77.4%
---------------------------------------------

👁️  Sample Artikel Berhasil Scrape (3 Pertama):
   [Negative] Cegah kolusi-nepotisme, program LPDP dinilai perlu perketat ... (2675 chars) — ANTARA News
   [Negative] LPDP Jakarta diminta tak ganggu anggaran bansos pendidikan -... (2304 chars) — ANTARA News
   [Negative] Dana LPDP dari pajak, penerima diminta hormati rakyat - ANTA... (1000 chars) —

## 💾 Export Dataset

Menyimpan **hanya artikel yang berhasil di-scrape** (punya `Content`) ke `dataset_lpdp_konten_raw.csv`.

> ⚠️ Artikel yang **gagal** di-scrape (Content kosong) **tidak diikutkan** — hanya artikel dengan konten lengkap yang dibawa ke Phase 3 selanjutnya.

### Struktur Kolom Output:
| Kolom | Keterangan |
|-------|------------|
| `Title` | Judul artikel |
| `Release Date` | Tanggal publish |
| `URL` | Google News URL (asli) |
| `Publisher` | Nama media/publisher |
| `PiC` | Person in Charge (anggota tim) |
| `Valid?` | Status validasi URL |
| `Sentiment` | Label sentimen (Positive/Neutral/Negative) |
| `Notes` | Catatan validasi manual |
| `Actual_URL` | URL asli setelah decode |
| `Content` | Isi lengkap artikel (hasil scraping) |

In [6]:
n_total_valid = len(df_valid)

# === Filter: hanya artikel yang punya Content ===
df_with_content = df_valid[df_valid["Content"].notna()].reset_index(drop=True).copy()
n_exported = len(df_with_content)
n_dropped  = n_total_valid - n_exported

print(f"🔍 Filter Konten:")
print(f"   Total artikel valid     : {n_total_valid}")
print(f"   Artikel punya content   : {n_exported} ({n_exported/n_total_valid*100:.1f}%) → DISIMPAN")
print(f"   Artikel tanpa content   : {n_dropped} ({n_dropped/n_total_valid*100:.1f}%) → DIKECUALIKAN")

# === Drop kolom helper sebelum export ===
cols_to_drop = ["content_len"]
export_df = df_with_content.drop(columns=[c for c in cols_to_drop if c in df_with_content.columns])

# === Simpan ke CSV ===
export_df.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Dataset berhasil disimpan ke: {OUTPUT_FILE}")
print(f"\n📋 Info File:")
print(f"   Rows   : {len(export_df):,}")
print(f"   Columns: {len(export_df.columns)}")
print(f"   Kolom  : {', '.join(export_df.columns.tolist())}")
print(f"   Ukuran : ~{export_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB (in-memory)")
print(f"\n📂 File output: {os.path.abspath(OUTPUT_FILE)}")

🔍 Filter Konten:
   Total artikel valid     : 1370
   Artikel punya content   : 1038 (75.8%) → DISIMPAN
   Artikel tanpa content   : 332 (24.2%) → DIKECUALIKAN

✅ Dataset berhasil disimpan ke: dataset_lpdp_konten_raw.csv

📋 Info File:
   Rows   : 1,038
   Columns: 10
   Kolom  : Title, Release Date, URL, Publisher, PiC, Valid?, Sentiment, Notes, Actual_URL, Content
   Ukuran : ~6.1 MB (in-memory)

📂 File output: c:\Users\mikba\Downloads\Documents\PBA\PBA\Project A\dataset_lpdp_konten_raw.csv


## ✅ Summary & Pipeline Checklist

Ringkasan akhir hasil proses scraping konten artikel LPDP.

In [7]:
print("="*60)
print("🎉 PIPELINE PHASE 2 — SUMMARY AKHIR")
print("="*60)

n_total_valid = len(df_valid)
n_decoded     = df_valid["Actual_URL"].notna().sum()
n_content     = len(export_df)           # sudah difilter → hanya yang punya content
n_dropped     = n_total_valid - n_content

print(f"\n📌 Proses yang Dilakukan:")
print(f"   1. Load dataset       ✅  {n_total_valid} artikel valid")
print(f"   2. Decode Google URL  ✅  {n_decoded}/{n_total_valid} berhasil ({n_decoded/n_total_valid*100:.1f}%)")
print(f"   3. Scrape content     ✅  {n_content}/{n_total_valid} artikel berhasil ({n_content/n_total_valid*100:.1f}%)")
print(f"   4. Export CSV         ✅  {OUTPUT_FILE} ({n_content} artikel punya content, {n_dropped} dikecualikan)")

print(f"\n📊 Distribusi Label (Artikel Diekspor = punya Content):")
for label in ["Positive", "Neutral", "Negative"]:
    n_lbl = (export_df["Sentiment"] == label).sum()
    print(f"   {label:<10}: {n_lbl:>4} artikel ({n_lbl/n_content*100:.1f}%)")

print(f"\n📁 Output File: {OUTPUT_FILE}")
print(f"   Total diekspor : {n_content} artikel (dari {n_total_valid} valid)")

print(f"\n" + "-"*60)
print("✅ CHECKLIST PHASE 2 (PIPELINE_LPDP_NLP.md)")
print("-"*60)

checks = [
    ("Import CSV ke Google Sheets",                      True),
    ("Validasi URL (True/False) per artikel",             True),
    ("Labeling manual 3 kelas",                           True),
    (f"{n_total_valid} artikel valid, 100% berlabel",     True),
    ("Decode Google News URL → Actual_URL",               n_decoded > 0),
    ("Scrape isi artikel → Content",                      n_content > 0),
    (f"Export {n_content} artikel punya content ke CSV",  True),
]

all_passed = True
for item, done in checks:
    status = "✅" if done else "⬜"
    if not done:
        all_passed = False
    print(f"  {status} {item}")

print(f"\n{'='*60}")
if all_passed:
    print(f"🎊 Phase 2 SELESAI! {n_content} artikel siap untuk Phase 3 (BERTopic) & Phase 4 (Preprocessing).")
else:
    print("⚠️  Ada item belum selesai. Cek output di atas.")
print("="*60)

print(f"\n➡️  Next Steps:")
print(f"   Phase 3 — BERTopic topic discovery (PIC: Celine)")
print(f"   Phase 4 — Preprocessing pipeline 10 langkah (PIC: Iqbal)")

🎉 PIPELINE PHASE 2 — SUMMARY AKHIR

📌 Proses yang Dilakukan:
   1. Load dataset       ✅  1370 artikel valid
   2. Decode Google URL  ✅  1305/1370 berhasil (95.3%)
   3. Scrape content     ✅  1038/1370 artikel berhasil (75.8%)
   4. Export CSV         ✅  dataset_lpdp_konten_raw.csv (1038 artikel punya content, 332 dikecualikan)

📊 Distribusi Label (Artikel Diekspor = punya Content):
   Positive  :  385 artikel (37.1%)
   Neutral   :  342 artikel (32.9%)
   Negative  :  311 artikel (30.0%)

📁 Output File: dataset_lpdp_konten_raw.csv
   Total diekspor : 1038 artikel (dari 1370 valid)

------------------------------------------------------------
✅ CHECKLIST PHASE 2 (PIPELINE_LPDP_NLP.md)
------------------------------------------------------------
  ✅ Import CSV ke Google Sheets
  ✅ Validasi URL (True/False) per artikel
  ✅ Labeling manual 3 kelas
  ✅ 1370 artikel valid, 100% berlabel
  ✅ Decode Google News URL → Actual_URL
  ✅ Scrape isi artikel → Content
  ✅ Export 1038 artikel punya con